In [ ]:
from pdbfixer import PDBFixer

from openmm.app import PDBFile
fixer = PDBFixer(filename='pdb_files/1F99.pdb')

fixer.findMissingResidues()
print('Missing Residues:', fixer.missingResidues)
fixer.findNonstandardResidues()
print('Nonstandard Residues:', fixer.nonstandardResidues)
fixer.replaceNonstandardResidues()
fixer.removeHeterogens()
fixer.findMissingAtoms()
print('Missing Atoms:', fixer.missingAtoms)
fixer.addMissingAtoms()
fixer.addMissingHydrogens(7.0)
PDBFile.writeFile(fixer.topology, fixer.positions, open('fixed_pdb_files/1A53.pdb', 'w'))

In [ ]:
target_chain = 'A'
chains_to_remove = []
for i, chain in enumerate(fixer.topology.chains()):
  if chain.id != target_chain:
    chains_to_remove.append(i)
  else:
    print(chain.id)

In [ ]:
import numpy as np
import biotite.structure.io as strucio
from pdbfixer import PDBFixer
from openmm.app import PDBxFile

def isolate_and_fix_chain(raw_id, input_cif, output_cif):
  """
  Uses PDBFixer to isolate a specific chain or entity, adds missing atoms, 
  and removes all other chains (including duplicate instances) from the structure.
  """
  print(f"Processing {raw_id}...")
  
  # 1. Resolve the Target Chain ID
  if "." in raw_id:
    pdb_id, target_chain = raw_id.split(".", 1)
    print(f"  -> Resolved PDB ID {pdb_id} to Chain {target_chain}")
  
  elif "_" in raw_id:
    pdb_id, entity_id = raw_id.split("_", 1)
    
    # PDBFixer doesn't read entity IDs, so we use Biotite to find the mapping
    array = strucio.load_structure(input_cif, model=1, extra_fields=["label_entity_id"])
    entity_mask = (array.label_entity_id == str(entity_id))
    
    if not np.any(entity_mask):
      raise ValueError(f"Entity ID {entity_id} not found in {input_cif}")
      
    # Grab the first chain associated with this entity
    target_chain = np.unique(array.chain_id[entity_mask])[0]
    print(f"  -> Resolved Entity {entity_id} to Chain {target_chain}")
    
  else:
    raise ValueError(f"Unrecognized ID format: {raw_id}")

  # 2. Load the structure into PDBFixer
  fixer = PDBFixer(filename=input_cif)

  # 3. Find indices of all chains EXCEPT the first instance of the target chain
  chains_to_remove = []
  kept_primary_chain = False
  
  # fixer.topology.chains() yields OpenMM Chain objects
  for i, chain in enumerate(fixer.topology.chains()):
    if chain.id == target_chain and not kept_primary_chain:
      # We found the first instance of our target chain. Keep it.
      kept_primary_chain = True
    else:
      # Delete everything else (other chains, and any duplicate/broken target chains)
      chains_to_remove.append(i)

  # Check if we accidentally marked everything for deletion
  if not kept_primary_chain:
    raise ValueError(f"Chain {target_chain} was not found in the PDBFixer topology!")
    
  print(f"  -> Removing {len(chains_to_remove)} other chain segment(s)...")
  fixer.removeChains(chains_to_remove)

  # 4. Fix the single, isolated chain
  print("  -> Adding missing heavy atoms...")
  fixer.findMissingResidues()
  fixer.findMissingAtoms()
  fixer.addMissingAtoms()

  # # 5. Save the perfectly clean, single-chain structure
  # with open(output_cif, 'w') as f:
  #   PDBxFile.writeFile(fixer.topology, fixer.positions, f)
    
  print(f"  -> Success! Saved to {output_cif}\n")

# Example Usage:
isolate_and_fix_chain("1F99.A", "cif_files/1F99.cif", "cif_files/1F99_fixed.cif")
isolate_and_fix_chain("1LMB_3", "cif_files/1LMB.cif", "cif_files/1LMB_fixed.cif")

In [ ]:
import pandas as pd

data_df = pd.read_csv("Morpheus_Supplement/output_results/training_data.csv")
for raw_id in tqdm(data_df['PDB_ID'], total=len(data_df)):
   """
  Uses PDBFixer to isolate a specific chain or entity, adds missing atoms, 
  and removes all other chains (including duplicate instances) from the structure.
  """
  tqdm.write(f"Processing {raw_id}...")
  try:
    # 1. Resolve the Target Chain ID
    if "." in raw_id:
      pdb_id, target_chain = raw_id.split(".", 1)
      tqdm.write(f"  -> Resolved PDB ID {pdb_id} to Chain {target_chain}")
      input_cif = f"cif_files/{pdb_id}.cif"
    elif "_" in raw_id:
      pdb_id, entity_id = raw_id.split("_", 1)
      input_cif = f"cif_files/{pdb_id}.cif"
      
      # PDBFixer doesn't read entity IDs, so we use Biotite to find the mapping
      array = strucio.load_structure(input_cif, model=1, extra_fields=["label_entity_id"])
      entity_mask = (array.label_entity_id == str(entity_id))
      
      if not np.any(entity_mask):
        raise ValueError(f"Entity ID {entity_id} not found in {input_cif}")
        
      # Grab the first chain associated with this entity
      target_chain = np.unique(array.chain_id[entity_mask])[0]
      tqdm.write(f"  -> Resolved Entity {entity_id} to Chain {target_chain}")
      
    else:
      raise ValueError(f"Unrecognized ID format: {raw_id}")
  
    # 2. Load the structure into PDBFixer
    fixer = PDBFixer(filename=input_cif)
  
    # 3. Find indices of all chains EXCEPT the first instance of the target chain
    chains_to_remove = []
    kept_primary_chain = False
    
    # fixer.topology.chains() yields OpenMM Chain objects
    for i, chain in enumerate(fixer.topology.chains()):
      if chain.id == target_chain and not kept_primary_chain:
        # We found the first instance of our target chain. Keep it.
        kept_primary_chain = True
      else:
        # Delete everything else (other chains, and any duplicate/broken target chains)
        chains_to_remove.append(i)
  
    # Check if we accidentally marked everything for deletion
    if not kept_primary_chain:
      raise ValueError(f"Chain {target_chain} was not found in the PDBFixer topology!")
      
    tqdm.write(f"  -> Removing {len(chains_to_remove)} other chain segment(s)...")
    fixer.removeChains(chains_to_remove)
  
    # 4. Fix the single, isolated chain
    tqdm.write("  -> Adding missing atoms...")
    fixer.findMissingResidues()
    fixer.findNonstandardResidues()
    fixer.replaceNonstandardResidues()
    fixer.removeHeterogens()
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(7.0)
  
    # 5. Save the perfectly clean, single-chain structure
    output_cif = f"fixed_cif_files/raw_id.cif"
    with open(output_cif, 'w') as f:
      PDBxFile.writeFile(fixer.topology, fixer.positions, f)
      
    tqdm.write(f"  -> Success! Saved to {output_cif}\n")
  except Exception as e:
    print(f"Failed to fix {filename}: {e}")
    

In [ ]:
fixer.